## The Noise groups

In [2]:
from astropy.io import fits
import numpy as np
from astropy.table import Table
import pandas as pd
import glob
from astropy.table import vstack
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.backends.backend_pdf import PdfPages

In [4]:
df_noise = pd.read_csv("../Class_wise_v4/Halpha_emitter_wise_noise.csv")
len(df_noise)

1750

In [6]:
for i in df_noise.columns:
    print(i)

Name
RAJ2000
DEJ2000
GLON
GLAT
SourceID
ePos
Class
pStar
pGalaxy
pNoise
imag
e_imag
imagAB
Elli
Classi
Deblendi
Saturatedi
Vignettedi
Traili
Truncatedi
BadPixi
MJDi
Seeingi
DetIDi
offRAi
offDEi
Hamag
e_Hamag
HamagAB
EllHa
ClassHa
DeblendHa
SaturatedHa
VignettedHa
TrailHa
TruncatedHa
BadPixHa
MJDHa
SeeingHa
DetIDHa
offRAHa
offDEHa
rImag
e_rImag
rImagAB
EllrI
ClassrI
DeblendrI
SaturatedrI
VignettedrI
TrailrI
TruncatedrI
BadPixrI
MJDrI
SeeingrI
DetIDrI
rUmag
e_rUmag
rUmagAB
EllrU
ClassrU
DeblendrU
SaturatedrU
VignettedrU
TrailrU
TruncatedrU
BadPixrU
MJDrU
SeeingrU
DetIDrU
offRArU
offDErU
gmag
e_gmag
gmagAB
Ellg
Classg
Deblendg
Saturatedg
Vignettedg
Trailg
Truncatedg
BadPixg
maskg
MJDg
Seeingg
DetIDg
offRAg
offDEg
Umag
e_Umag
EllU
ClassU
DeblendU
SaturatedU
VignettedU
TrailU
TruncatedU
BadPixU
MJDU
SeeingU
DetIDU
offRAU
offDEU
brightN
deblend
saturated
nBands
errBits
nObsI
nObsU
FieldIDI
FieldIDU
FieldGradeI
FieldGradeU
emitter
variable
SourceID2
imag2
e_imag2
Classi2
Seeingi2
MJDi2
offRAi

In [37]:
# Ejemplo: Errores fotométricos < 0.2 mag
good_quality = (
    (df_noise['e_Jmag'] < 0.2) &
    (df_noise['e_Hmag'] < 0.2) &
    (df_noise['e_Kmag'] < 0.2) &
    (df_noise['e_W1mag'] < 0.2) &
    (df_noise['e_W2mag'] < 0.2) &
    (df_noise['e_W3mag'] < 0.5)
)
df_noise_clean = df_noise[good_quality]
len(df_noise_clean)

1472

In [71]:
# 1. Calcular colores necesarios (en sistema Vega)
df_noise_clean['H_W2'] = df_noise_clean['Hmag'] - df_noise_clean['W2mag']
df_noise_clean['Ks_W3'] = df_noise_clean['Kmag'] - df_noise_clean['W3mag']
df_noise_clean['J_H'] = df_noise_clean['Jmag'] - df_noise_clean['Hmag']
df_noise_clean['W1_W2'] = df_noise_clean['W1mag'] - df_noise_clean['W2mag']
df_noise_clean['W1_W4'] = df_noise_clean['W1mag'] - df_noise_clean['W4mag']
df_noise_clean['H_W2'] = df_noise_clean['Hmag'] - df_noise_clean['W2mag']
df_noise_clean['Ks_W3'] = df_noise_clean['Kmag'] - df_noise_clean['W3mag']

/tmp/ipykernel_14429/950808865.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_noise_clean['H_W2'] = df_noise_clean['Hmag'] - df_noise_clean['W2mag']
/tmp/ipykernel_14429/950808865.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_noise_clean['Ks_W3'] = df_noise_clean['Kmag'] - df_noise_clean['W3mag']
/tmp/ipykernel_14429/950808865.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cav

In [81]:
# 4. Aplicar criterios de selección CORRECTOS
# Condición base común
cond_universal = (
   df_noise_clean['H_W2'] >= 0.206) & (df_noise_clean['Ks_W3'] >= 0.27
)

In [87]:
# 3. Filtrar candidatos
syst_candidates = df_noise_clean[cond_universal].copy()

In [88]:
# 4. Limpieza básica de datos (opcional pero recomendado)
# - Eliminar objetos sin magnitudes críticas
syst_candidates = syst_candidates.dropna(subset=['Hmag', 'W2mag', 'Kmag', 'W3mag', 'Jmag', 'Hmag'])

In [89]:
# 5. Seleccionar columnas clave para crossmatch
cols_to_keep = ['Name', 'RAJ2000', 'DEJ2000', 'Jmag', 'Hmag', 'Kmag', 'W1mag', 'W2mag', 'W3mag', 'H_W2', 'Ks_W3', 'J_H']
syst_candidates = syst_candidates[cols_to_keep]

In [90]:
len(syst_candidates)

1406

In [91]:
syst_candidates

,Name,RAJ2000,DEJ2000,Jmag,Hmag,Kmag,W1mag,W2mag,W3mag,H_W2,Ks_W3,J_H
1,J193811.37+314430.9,294.547359,31.741924,13.849,13.173,13.073,12.877,12.947,12.430,0.226,0.643,0.676
2,J195003.04+320607.2,297.512666,32.101992,14.570,13.612,12.863,11.782,11.190,9.876,2.422,2.987,0.958
3,J195655.87+315637.0,299.232804,31.943608,13.036,12.192,11.584,10.849,10.301,6.964,1.891,4.620,0.844
4,J195713.05+320747.4,299.304390,32.129837,13.564,12.970,12.530,12.122,11.869,11.463,1.101,1.067,0.594
5,J195714.52+322240.7,299.310482,32.377976,13.589,12.987,12.460,11.795,11.432,10.912,1.555,1.548,0.602
...,...,...,...,...,...,...,...,...,...,...,...,...
1743,J200318.85+311503.5,300.828544,31.250981,13.007,12.614,12.358,12.052,11.783,10.037,0.831,2.321,0.393
1745,J195837.16+312322.7,299.654844,31.389637,15.497,14.847,14.181,12.788,11.198,6.859,3.649,7.322,0.650
1746,J193948.01+285751.9,294.950056,28.964425,14.426,13.760,13.587,13.397,13.516,11.922,0.244,1.665,0.666
1748,J194631.32+291451.9,296.630484,29.247751,13.798,13.217,12.870,12.160,11.816,10.431,1.401,2.439,0.581


In [92]:
# 9. Guardar para crossmatch con LAMOST
syst_candidates.to_csv('../Class_wise_v4/SySt_S-type_candidates_Noise.csv', index=False)